# Validate the Python port of the lab's processing

The lab computes the processed fiber photometry signals of each TDT block with MATLAB code: the raw photodetector signal is demodulated at the modulation frequency of each LED, resampled to 100 Hz, and dF/F is computed against a bi-exponential fit of the isosbestic signal scaled to the calcium signal. `hadjas_2025.analysis.processing` ports these steps to Python.

This notebook recomputes the processed signals of one session from its raw signal and compares them with the lab's, then summarises the same comparison over all sessions. Demodulation and resampling give the lab's signals to floating point precision. dF/F differs where the bi-exponential fit differs from MATLAB's; the docstring of the `processing` module explains why.

## 1. Open a session

The NWB file must hold the raw signal and the lab's processed signals, as written by `hadjas-2025 convert <data folder> <output folder>` with the default `processed: lab` of the session config. Once the files of the current conversion are on the DANDI draft, a session can also be streamed with `load_nwb_from_dandi` (commented out below).

In [ ]:
from pathlib import Path
import re

import pandas as pd
from pynwb import NWBHDF5IO

from hnasko_lab_to_nwb.hadjas_2025.analysis.lab_comparison import compare_signals, plot_session_comparison
from hnasko_lab_to_nwb.hadjas_2025.analysis.processed_series import compute_processed_series, get_series_blocks
from hnasko_lab_to_nwb.hadjas_2025.analysis.processing import PROCESSED_STREAM_NAMES
from hnasko_lab_to_nwb.hadjas_2025.convert_session import get_led_modulation, load_cohort_metadata

# An NWB file with the raw and the lab's processed signals, and the cohort of its subject (a key of "cohorts" in
# metadata/session_config.yaml)
nwbfile_path = Path("nwb_files/sub-C4561_ses-varying_frequencies.nwb")
recording_type = "SN pan GABA recordings"

io = NWBHDF5IO(nwbfile_path, "r")
nwbfile = io.read()
# from hnasko_lab_to_nwb.hadjas_2025.analysis.dandi_streaming import load_nwb_from_dandi
# nwbfile, io = load_nwb_from_dandi("001528", "C4561", "varying-frequencies")
print(nwbfile.session_description)

## 2. Recompute the processed signals

The raw series is in `nwbfile.acquisition`. A session of several TDT blocks is written as one series with timestamps, which is split again at the gaps between blocks; each block is processed separately, as in the lab's code. The modulation frequencies of the two LEDs depend on the recording date (`led_modulation` in the session config).

In [ ]:
metadata = load_cohort_metadata(recording_type)
raw_metadata_key = next(
    key
    for key, series_metadata in metadata["FiberPhotometry"].items()
    if key.startswith("Fi1r_") and series_metadata["name"] in nwbfile.acquisition
)
site = raw_metadata_key.removeprefix("Fi1r_")
raw_series = nwbfile.acquisition[metadata["FiberPhotometry"][raw_metadata_key]["name"]]
led_modulation = get_led_modulation(nwbfile.session_start_time)
blocks = compute_processed_series(raw_series=raw_series, led_modulation=led_modulation)
print(f"Raw series {raw_series.name}, LED modulation {led_modulation}")
print(f"Start of each block: {[round(float(block['starting_time']), 1) for block in blocks]} s")

## 3. Compare with the lab's signals

Each of the lab's series is split into the same blocks. For each block and stream, `compare_signals` gives the correlation, the largest absolute difference and the standard deviation of the lab's signal over the samples both signals have. The last column is the largest difference relative to that standard deviation.

In [ ]:
ophys = nwbfile.processing["ophys"]
lab_blocks = {}
for stream_name in PROCESSED_STREAM_NAMES:
    lab_series = ophys[metadata["FiberPhotometry"][f"{stream_name}_{site}"]["name"]]
    block_bounds, _, _ = get_series_blocks(lab_series)
    data = lab_series.data[:]
    lab_blocks[stream_name] = [data[start:end] for start, end in block_bounds]
    assert len(block_bounds) == len(blocks), f"{lab_series.name} has {len(block_bounds)} blocks, not {len(blocks)}"

rows = []
for index, block in enumerate(blocks):
    for stream_name in PROCESSED_STREAM_NAMES:
        comparison = compare_signals(recomputed=block[stream_name], lab=lab_blocks[stream_name][index])
        rows.append(dict(block=index + 1, stream=stream_name, **comparison))
session_comparison = pd.DataFrame(rows)
session_comparison["relative_difference"] = (
    session_comparison["max_absolute_difference"] / session_comparison["lab_standard_deviation"]
)
session_comparison

Left, the 405 nm signal and the recomputed bi-exponential fit of each block (the lab's fit is in its .mat files and not in the NWB file; `hadjas-2025 compare-processing --plot-dir-path` plots both). Right, the lab's and the recomputed dF/F.

In [ ]:
plot_session_comparison(
    blocks=[
        dict(
            block=f"block {index + 1}",
            af=block["af"],
            af_fit=block["af_fit"],
            dF=block["dF"],
            lab_dF=lab_blocks["dF"][index],
        )
        for index, block in enumerate(blocks)
    ],
    title=f"{nwbfile.subject.subject_id}, {nwbfile.session_id}: lab (MATLAB) and recomputed (Python)",
)

In [ ]:
io.close()

## 4. All sessions

`hadjas-2025 compare-processing <data folder> <output folder>` makes the same comparison for every block of every session in the lab's data share, against the lab's .mat files, and writes `processing_comparison.csv` with one row per block and stream. It needs the data share, so it is run separately; set the path to its output below.

The table counts, per cohort and stream, the blocks whose largest difference is below 1 % of the standard deviation of the lab's signal. Blocks whose fit the lab adjusted by hand (`LAB_ADJUSTED_FITS` in `lab_comparison`) are left out of it and listed below, since their dF/F is not expected to match.

In [ ]:
comparison_file_path = Path("processing_comparison.csv")

all_comparisons = pd.read_csv(comparison_file_path)
compared = all_comparisons[all_comparisons["error"].isna()].copy()
compared["cohort"] = compared["protocol_folder_path"].map(
    lambda path: next(part for part in re.split(r"[\\/]", path) if part.endswith("recordings"))
)
compared["relative_difference"] = compared["max_absolute_difference"] / compared["lab_standard_deviation"]
print(
    f"{compared.groupby(['subject_id', 'protocol_folder_path']).ngroups} sessions, {len(compared) // len(PROCESSED_STREAM_NAMES)} blocks"
)

compared[compared["lab_adjusted_fit"].isna()].groupby(["cohort", "stream"]).agg(
    blocks=("relative_difference", "size"),
    median_correlation=("correlation", "median"),
    blocks_within_1_percent=("relative_difference", lambda difference: int((difference < 0.01).sum())),
)

In [ ]:
compared[compared["lab_adjusted_fit"].notna() & (compared["stream"] == "dF")][
    ["subject_id", "protocol", "block", "segment", "lab_adjusted_fit", "correlation"]
]

Sessions whose comparison failed, for example because the lab's .mat file holds another recording than the data share's blocks:

In [ ]:
all_comparisons[all_comparisons["error"].notna()][["subject_id", "protocol", "error"]]